# THINGS-EEG2: shared data loader, within-subject setting

Turns the raw THINGS-EEG2 files into model-ready arrays, **identically for the whole team**. The split and the preprocessing numbers are **embedded in this notebook** (cell 2), so no extra files are needed. Both loader notebooks (within-subject and LOSO) carry the same config; the printed **config id must be `03e3ad5453bd`** in every notebook. If it differs, someone edited the numbers.

How it was made: `THINGS_EEG2_split_and_preprocessing.ipynb` (165 validation concepts = 10%, seed 0; per-subject robust statistics fitted on training concepts only; clip K = 6 robust sd; 17 posterior channels; window 0-590 ms = 60 points). Each trial: median removed, divided by the robust scale, clipped to +-6; repetitions are averaged afterwards.

**`within_subject(s)`:** train = subject `s` on training concepts, validation = subject `s` on validation concepts (1,650 images), test = subject `s` on the 200 test images. Normalisation uses the subject's own statistics.

**How to use**
1. Set `REPO_ID` in cell 1 (org/repo of the dataset) and add the Colab Secret `HF_TOKEN`.
2. Run cells 1-3. In your own training notebook, copy cells 1-3 to the top.
3. Call the function (examples at the bottom).

**Image order (needed for the targets / embeddings):** `idx_train` and `idx_val` are row numbers of the 16,540 training images in the order of `image_metadata.npy` (sorted by concept, 10 images per concept); test rows are the 200 test images in the order of `TEST_FILES`. Test data are only returned with `include_test=True`; use it for the final evaluation only.

In [ ]:
import os, sys, json, gc, pickle, subprocess, time
import numpy as np

# ------------------------------ config ------------------------------
REPO_ID = os.environ.get('HF_REPO_ID', 'things-eeg2/things-eeg2-preprocessed') 
LOCAL_ROOT = os.environ.get('EDA_LOCAL_ROOT')                    # None on Colab -> download from the Hub
WORK = os.environ.get('EDA_WORK', '/content/work')
os.makedirs(WORK, exist_ok=True)


if not LOCAL_ROOT:
    assert 'PUT_ORG_NAME' not in REPO_ID, 'set REPO_ID (org/repo) in this cell'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=False)
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = os.environ.get('HF_TOKEN')
    if not HF_TOKEN:
        from getpass import getpass
        HF_TOKEN = getpass('Hugging Face token (read access): ')
    from huggingface_hub import hf_hub_download

def get_file(rel):
    if LOCAL_ROOT:
        return os.path.join(LOCAL_ROOT, rel)
    return hf_hub_download(REPO_ID, rel, repo_type='dataset', token=HF_TOKEN, local_dir=WORK)

def release(path):
    if not LOCAL_ROOT and os.path.exists(path):
        os.remove(path)

# restricted unpickler: the .npy files are pickled dicts, only NumPy types are allowed
_OK = {('numpy', 'ndarray'), ('numpy', 'dtype'),
       ('numpy.core.multiarray', '_reconstruct'), ('numpy._core.multiarray', '_reconstruct'),
       ('numpy.core.multiarray', 'scalar'), ('numpy._core.multiarray', 'scalar'),
       ('builtins', 'dict'), ('builtins', 'list')}
class _U(pickle.Unpickler):
    def find_class(self, module, name):
        if (module, name) in _OK:
            return getattr(__import__(module, fromlist=[name]), name)
        raise pickle.UnpicklingError(f'blocked {module}.{name}')
def safe_load(path):
    obj = np.load(path, allow_pickle=True)
    return obj.item() if isinstance(obj, np.ndarray) and obj.shape == () else obj
print('setup ok | local mode =', bool(LOCAL_ROOT))

In [ ]:
# ============================== EMBEDDED CONFIG (same in all loader notebooks, do not edit) ==============================
EMBEDDED = {
  'config_id': '03e3ad5453bd',
  'source': '17ch', 'clip_k': 6.0, 'crop_ms': [0, 590], 'n_time_points': 60,
  'channels': ['Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2'],
  'validation_concepts': ['00005_air_conditioner', '00009_airboat', '00013_almond', '00025_anteater', '00027_anvil', '00034_arch', '00044_axe', '00051_badger', '00062_bandage', '00077_basketball_hoop', '00080_bat2', '00082_bathrobe', '00084_baton1', '00094_beanie', '00113_binder', '00115_bird', '00123_blazer', '00127_blinder', '00134_blueberry', '00136_boar', '00147_book', '00172_branch', '00191_bulldozer', '00211_cake_mix', '00241_caramel', '00263_cereal', '00268_chalkboard', '00321_cloud', '00324_coal', '00360_corkboard', '00371_cotton_candy', '00382_crayfish', '00396_crystal_ball', '00401_cup', '00403_curling_iron', '00418_desk', '00419_detonator', '00458_dryer', '00460_duckling', '00484_emerald', '00485_enchilada', '00504_fern', '00507_fig', '00519_fire_pit', '00533_flamethrower', '00538_flatiron', '00562_frame', '00575_garbage', '00586_gear', '00587_gearshift', '00592_ginger', '00595_girl', '00597_glasses', '00599_glue', '00604_goggles', '00617_grapevine', '00619_grate', '00638_gun', '00647_hairnet', '00652_hamster', '00661_harmonica', '00670_headband', '00674_headphones', '00696_honeypot', '00698_hook1', '00711_hotplate', '00721_ice-cream_cone', '00722_ice_cream', '00736_ivy', '00757_jump_rope', '00759_kale', '00760_kaleidoscope', '00763_kazoo', '00776_knife', '00795_latte', '00808_lemonade', '00820_lighter', '00823_lingerie', '00826_lip_gloss', '00832_locker', '00834_log', '00841_lunchbox', '00845_machine_gun', '00898_mold2', '00901_mongoose', '00912_mousepad', '00918_muffin', '00937_nest', '00939_nightshirt', '00948_oar', '00950_octopus', '00953_oilcan', '00975_palm_tree', '00988_paperclip', '00994_pasta', '00995_pastry', '01008_peeler', '01015_penguin', '01020_pepper_mill', '01027_petal', '01030_phone_booth', '01037_piggy_bank', '01057_place_mat', '01061_plate', '01068_plunger', '01073_polar_bear', '01090_potato', '01093_potpie', '01098_printer', '01109_pump', '01124_rack2', '01125_racket', '01132_rain_gauge', '01133_raincoat', '01162_rifle', '01177_rollerskate', '01201_sandcastle', '01220_school_bus', '01232_screwdriver', '01233_scuba', '01234_sea_urchin', '01243_sewage', '01247_shark', '01252_sheep', '01253_sheet', '01262_shoe_polish', '01268_shorts', '01279_sickle', '01287_skeleton', '01299_slime', '01303_slot_machine', '01315_snowman', '01318_snowsuit', '01324_sofa_bed', '01360_squirt_gun', '01371_starfish', '01378_stereo', '01393_straightjacket', '01398_streetlight', '01408_sundial', '01412_surfboard', '01436_taco', '01445_tape_measure', '01455_teapot', '01460_telephone_pole', '01462_tennis_ball', '01464_terrarium', '01471_thread', '01515_trampoline', '01516_trap', '01528_trolley', '01531_trough', '01539_tumbleweed', '01553_ukulele', '01559_urn', '01563_vase', '01606_wedding_cake', '01607_wedge', '01610_whale', '01616_whistle', '01637_worm', '01638_wrap', '01639_wrapping_paper', '01647_yarn', '01650_yoke'],
  'own': {
    'sub-01': {'median': [0.05212551727890968, -0.0412634052336216, -0.012548509985208511, -0.060622379183769226, 0.0031204361002892256, -0.05223836749792099, 0.0016748294001445174, 0.006696650758385658, -0.00018802081467583776, -0.047741010785102844, -0.0882076546549797, -0.08574133366346359, 0.05044333636760712, -0.05083566904067993, -0.0799252837896347, -0.024858374148607254, 0.040171027183532715], 'scale': [0.8270606398582458, 0.7572140097618103, 0.2860817015171051, 0.6828564405441284, 0.9382297396659851, 0.7311781644821167, 0.6771243810653687, 0.8701529502868652, 0.7186931371688843, 0.7728356719017029, 0.720960795879364, 0.8259994983673096, 0.8881974220275879, 0.7313801646232605, 0.7164547443389893, 0.7481438517570496, 0.6912838220596313]},
    'sub-02': {'median': [0.03659636154770851, 0.006098971702158451, -0.051767874509096146, -0.07265397161245346, 0.04467694088816643, -0.0669691190123558, 0.03918806463479996, -0.0754278153181076, 0.04087003692984581, 0.01481194980442524, -0.10776939988136292, -0.09100008010864258, -0.02567923441529274, -0.05861956626176834, -0.14563879370689392, -0.04860378056764603, -0.0015662352088838816], 'scale': [0.8430904746055603, 0.6887317895889282, 0.8255415558815002, 0.6620797514915466, 0.7978435158729553, 0.6278082728385925, 0.7351393699645996, 0.7699784636497498, 0.7420060634613037, 0.6756360530853271, 0.6915464997291565, 0.6828659176826477, 0.7100948095321655, 0.6355234980583191, 0.7367954254150391, 0.7041008472442627, 0.7865579128265381]},
    'sub-03': {'median': [-0.007543467450886965, -0.04561395198106766, -0.0384070947766304, -0.04645868390798569, 0.017624858766794205, -0.007268466055393219, -0.04148063808679581, -0.03872350603342056, -0.0384550578892231, -0.043929003179073334, -0.042220063507556915, -0.050546322017908096, -0.008769691921770573, -0.06036440283060074, -0.08763857185840607, -0.06580780446529388, -0.0027134688571095467], 'scale': [0.7132353186607361, 0.5588681101799011, 0.6652225852012634, 0.6796180605888367, 0.8595502376556396, 0.8638209700584412, 0.6407350301742554, 0.8119226694107056, 0.5827350616455078, 0.5583130717277527, 0.6418108344078064, 0.5677142143249512, 0.6989087462425232, 0.6608988642692566, 0.7730374932289124, 0.7375794053077698, 0.7215368747711182]},
    'sub-04': {'median': [-0.009533200412988663, -0.04599408060312271, -0.07197532057762146, -0.08895833045244217, -0.08716186881065369, -0.10945571959018707, -0.07654144614934921, -0.11458441615104675, -0.028006164357066154, -0.06302910298109055, -0.019957415759563446, -0.042703595012426376, -0.039295509457588196, -0.08972638845443726, -0.12975487112998962, -0.10493022203445435, -0.016184117645025253], 'scale': [0.7159298062324524, 0.5306655764579773, 0.6119266152381897, 0.6120895743370056, 0.7430707216262817, 0.6357284188270569, 0.6577863097190857, 0.6540287137031555, 0.6244242191314697, 0.5785959362983704, 0.2420497089624405, 0.6280364394187927, 0.6927803754806519, 0.6423023343086243, 0.6462635397911072, 0.6537368893623352, 0.6952630281448364]},
    'sub-05': {'median': [-0.02107340469956398, -0.04638443887233734, -0.06189097464084625, -0.045009639114141464, -0.030699273571372032, -0.06768260896205902, -0.047264084219932556, -0.049122001975774765, -0.0172785222530365, -0.046344246715307236, -0.07267837226390839, -0.04648732393980026, -0.024266716092824936, -0.0641937106847763, -0.07232839614152908, -0.05439621955156326, -0.020080404356122017], 'scale': [0.8556995391845703, 0.7107377052307129, 0.8497781157493591, 0.8539704084396362, 0.8947201371192932, 0.8253223896026611, 0.8748919367790222, 0.8527870178222656, 0.9104912877082825, 0.7314068078994751, 0.8514584898948669, 0.8832912445068359, 0.81545490026474, 0.8046797513961792, 0.8195454478263855, 0.8491811752319336, 0.82081139087677]},
    'sub-06': {'median': [0.05259566754102707, 0.033518776297569275, -0.0054082018323242664, -0.047632135450839996, -0.013038523495197296, -0.03836918994784355, 0.023883188143372536, 0.04512272775173187, -0.0006860482390038669, -0.0019537112675607204, -0.05821384862065315, -0.01911746896803379, -0.026366978883743286, -0.05772998183965683, -0.05454442277550697, -0.012105927802622318, 0.004831115249544382], 'scale': [0.8939808011054993, 0.7746798992156982, 0.8158217668533325, 0.7379837036132812, 0.9241583943367004, 0.7340677976608276, 0.8400367498397827, 0.92352694272995, 0.7490238547325134, 0.7318480610847473, 0.843039333820343, 0.7701483368873596, 0.7493153214454651, 0.7899198532104492, 0.8461984992027283, 0.8013942837715149, 0.8220378756523132]},
    'sub-07': {'median': [0.04533875733613968, 0.0008466507424600422, -0.0034844265319406986, -0.03624957054853439, 0.05738560110330582, -0.015028152614831924, -0.01949882134795189, -0.03508397191762924, 0.028056923300027847, -0.0029636535327881575, -0.021598054096102715, -0.03533262014389038, 0.022155724465847015, -0.0950155109167099, -0.06369851529598236, -0.03882923722267151, 0.02253328636288643], 'scale': [0.7265564203262329, 0.5907326340675354, 0.6514288783073425, 0.7102640271186829, 1.0412696599960327, 0.863789975643158, 0.7553276419639587, 0.7981641888618469, 0.6711799502372742, 0.5980116128921509, 0.7149447202682495, 0.7126977443695068, 0.7355523705482483, 0.8172610998153687, 0.7377721667289734, 0.7001919150352478, 0.7096966505050659]},
    'sub-08': {'median': [-0.03150847926735878, -0.049703553318977356, -0.08279313147068024, -0.040952667593955994, 0.009480603039264679, -0.08917061984539032, -0.0943540409207344, -0.06327603757381439, -0.0280262790620327, -0.08150352537631989, -0.0989578366279602, -0.0654364675283432, -0.045882366597652435, -0.15948686003684998, -0.11215551197528839, -0.09046090394258499, -0.05693255364894867], 'scale': [0.7100861668586731, 0.6501478552818298, 0.8336851596832275, 0.7609663605690002, 0.8932704925537109, 0.6892245411872864, 0.7120484113693237, 0.8657137751579285, 0.7007925510406494, 0.7047814130783081, 0.6715712547302246, 0.7108237743377686, 0.7178791761398315, 0.7948876619338989, 0.743849515914917, 0.8375179767608643, 0.7418733239173889]},
    'sub-09': {'median': [0.012050089426338673, -0.02955237589776516, -0.02426966279745102, -0.08258766680955887, -0.05234565585851669, -0.07195344567298889, -0.022472210228443146, -0.03747932240366936, -0.004361548461019993, -0.047348856925964355, -0.029814377427101135, -0.05512122064828873, -0.007468471769243479, -0.048780642449855804, -0.03617263212800026, -0.04620153456926346, 0.0027649137191474438], 'scale': [0.7336651682853699, 0.7121814489364624, 0.911578893661499, 0.6726694703102112, 0.7461410760879517, 0.7334755063056946, 0.6821542382240295, 0.8942809104919434, 0.6938963532447815, 0.7198354601860046, 0.8513979315757751, 0.6618514060974121, 0.6696056127548218, 0.6976416707038879, 0.9187774062156677, 0.6991885900497437, 0.7454212307929993]},
    'sub-10': {'median': [-0.013435076922178268, -0.03593704476952553, -0.033917687833309174, -0.06176971644163132, -0.05961417406797409, -0.03501330688595772, -0.02591799572110176, -0.03494369238615036, -0.02489575371146202, -0.03572196513414383, -0.08436740934848785, -0.05253987014293671, -0.0029226099140942097, -0.05198246240615845, -0.07541310787200928, -0.023481806740164757, -0.018681086599826813], 'scale': [0.6683067679405212, 0.656835675239563, 0.8738408088684082, 0.7280109524726868, 1.0712761878967285, 0.8694833517074585, 0.6623878479003906, 0.868741512298584, 0.6638216972351074, 0.6942368149757385, 0.8164786696434021, 0.6766190528869629, 0.7017368674278259, 0.6495623588562012, 0.7979832291603088, 0.6736092567443848, 0.6620151400566101]},
  },
}

In [ ]:
# ============================== LIBRARY (copy cells 1-3 into your own notebook) ==============================

CFG = EMBEDDED
assert CFG['source'] == '17ch', 'the statistics were fitted on the 17-channel files; refit them for another channel set'
CH17 = CFG['channels']; CLIP_K = float(CFG['clip_k']); CROP_MS = CFG['crop_ms']
TS = np.arange(100) * 0.01 - 0.2                                 # -0.20 ... 0.79 s, stimulus onset at 0
CROP = (TS >= CROP_MS[0] / 1000 - 1e-9) & (TS <= CROP_MS[1] / 1000 + 1e-9)
assert int(CROP.sum()) == CFG['n_time_points']

_meta = safe_load(get_file('images/image_metadata.npy'))
TRAIN_CONCEPTS = np.array(_meta['train_img_concepts']); assert list(TRAIN_CONCEPTS) == sorted(TRAIN_CONCEPTS)
TRAIN_FILES = list(zip(TRAIN_CONCEPTS, _meta['train_img_files']))        # (concept folder, file name) per EEG row
TEST_FILES = list(zip(_meta['test_img_concepts'], _meta['test_img_files']))
IS_VAL = np.isin(TRAIN_CONCEPTS, CFG['validation_concepts'])
TRAIN_IDX = np.flatnonzero(~IS_VAL); VAL_IDX = np.flatnonzero(IS_VAL)
assert len(VAL_IDX) == 10 * len(CFG['validation_concepts']) and len(TRAIN_IDX) + len(VAL_IDX) == 16540

def _stats(d): return dict(median=np.asarray(d['median'], np.float32), scale=np.asarray(d['scale'], np.float32))

def preprocess(x, stats, k=CLIP_K, crop=True):
    """x: (..., 17, 100) raw trials -> (..., 17, 60): crop, remove median, divide by robust scale, clip to +-k."""
    if crop: x = x[..., CROP]
    z = (x - stats['median'][:, None]) / stats['scale'][:, None]
    return np.clip(z, -k, k).astype(np.float32)

def load_eeg(s, split):
    p = get_file(f'eeg/17ch/{s}/preprocessed_eeg_{split}.npy'); T = safe_load(p); release(p)
    X = T['preprocessed_eeg_data']; assert T['ch_names'] == CH17 and X.shape[2:] == (17, 100)
    return X

def _prep_rows(Y, rows, stats, avg_reps, chunk=2000):
    out = []
    for a in range(0, len(rows), chunk):
        z = preprocess(Y[rows[a:a + chunk]], stats)
        out.append(z.mean(1) if avg_reps else z)
    return np.concatenate(out)

def subject_data(s, stats, parts=('train', 'val'), avg_reps=True):
    """Preprocess subject s with the given statistics. parts: any of 'train', 'val', 'test'."""
    d = {}
    if 'train' in parts or 'val' in parts:
        Y = load_eeg(s, 'training'); assert Y.shape == (16540, 4, 17, 100), Y.shape
        if 'train' in parts: d['X_train'] = _prep_rows(Y, TRAIN_IDX, stats, avg_reps); d['idx_train'] = TRAIN_IDX
        if 'val' in parts: d['X_val'] = _prep_rows(Y, VAL_IDX, stats, avg_reps); d['idx_val'] = VAL_IDX
        del Y; gc.collect()
    if 'test' in parts:
        X = load_eeg(s, 'test'); assert X.shape == (200, 80, 17, 100), X.shape
        d['X_test'] = _prep_rows(X, np.arange(200), stats, avg_reps); del X
    return d

def within_subject(s, avg_reps=True, include_test=False):
    """Within-subject setting for subject s (the subject's own statistics).
    Returns X_train (n, 17, 60), idx_train, X_val, idx_val [, X_test (200, 17, 60)].
    avg_reps=False keeps the repetitions: (n, 4, 17, 60) for train/val and (200, 80, 17, 60) for test."""
    return subject_data(s, _stats(CFG['own'][s]), ('train', 'val') + (('test',) if include_test else ()), avg_reps)

SUBJECTS = list(CFG['own'])
print('loader ready | config id', CFG['config_id'], '(must be identical in every notebook of the team) |', len(SUBJECTS), 'subjects | channels:', len(CH17),
      '| window:', CROP_MS, 'ms |', int(CROP.sum()), 'time points | clip_k =', CLIP_K, '| train images', len(TRAIN_IDX), '| validation images', len(VAL_IDX))

## Self-test
Checks the shapes, no NaN, values inside the clip range, no overlap between the sets, and that no validation concept is in the training rows.

In [ ]:
s0 = SUBJECTS[0]; t = time.time()
d = within_subject(s0)
print(s0, {k: (v.shape if hasattr(v, 'shape') else v) for k, v in d.items()}, f'{time.time() - t:.0f}s')
assert d['X_train'].shape == (len(TRAIN_IDX), 17, 60) and d['X_val'].shape == (len(VAL_IDX), 17, 60)
for k in ('X_train', 'X_val'):
    assert not np.isnan(d[k]).any() and np.abs(d[k]).max() <= CLIP_K + 1e-4
assert not set(d['idx_train']) & set(d['idx_val'])
assert not set(TRAIN_CONCEPTS[d['idx_train']]) & set(TRAIN_CONCEPTS[d['idx_val']]), 'concept leaked between train and validation'
assert 'X_test' not in d
dw = within_subject(s0, avg_reps=False); assert dw['X_val'].shape == (len(VAL_IDX), 4, 17, 60); del dw
print('within-subject ok: std train %.3f, std val %.3f (should be similar)' % (d['X_train'].std(), d['X_val'].std()))